# Case Ambev — Long Neck NENO | **Planejamento Março**

**Papel:** Especialista de Planejamento  
**Data-base:** 02/02/2026  
**Objetivo:** Simular a viabilidade de atender +10% de demanda em março via cabotagem embarcada em fevereiro

> **SKUs analisados:** Malzbier Brahma | Goose Island | Colorado  
> **Patagonia:** excluída (sem rota de transferência SE→NENO)  
> **BIAS −9%:** aplicado em todos os SKUs  
> **Modal:** cabotagem — embarca 02/02, lead 25 dias, chega ~27/02 (W0 de março)  
> **Avaria:** não se aplica à cabotagem — embarque = chegada


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 130)

semanas = ['W0_02fev', 'W1_09fev', 'W2_16fev', 'W3_23fev']
DOI_MIN = 12
CAP_AQ  = 12600
CAP_NS  = 27000

## 1. Premissas do Modelo

| Parâmetro | Valor | Justificativa |
|---|---|---|
| Crescimento demanda | +10% | Sinalização GEOs NE e NO |
| Base demanda março | Dem fev c/BIAS × 1,10 ÷ 4 | Distribuição uniforme entre 4 semanas |
| BIAS | −9% (÷ 1,09) | GEOs superestimam sistematicamente |
| EI março | EF W3 fevereiro | Sequência natural do planejamento |
| Modal | Cabotagem | Único viável — lead 25d, embarca 02/02, chega ~27/02 |
| Avaria | **Não aplicável** | Cabotagem não tem avaria — embarque = chegada |
| DOI mínimo | 12 dias | Política NENO |
| Simulação | Apenas W0 de março | PCP de março desconhecido |
| Split BA/PB | Proporção demanda W0 fev | Semana de referência com transferência |
| NO Araguaia | Excluído | Retirada em UB541/MG |


## 2. Dados de Entrada — Demanda Fevereiro e EI Março


In [2]:
XLS = 'Analise_LongNeck_WSNP - Sem repostas.xlsx'
df_nd = pd.read_excel(XLS, sheet_name='Cenário com Nova Demanda', header=None)
df_ct = pd.read_excel(XLS, sheet_name='Custos de transferência',  header=None)

_SC = {
    'W0': {'dem': 3,  'ei':  7, 'ef': 13},
    'W1': {'dem': 16,           'ef': 24},
    'W2': {'dem': 27,           'ef': 35},
    'W3': {'dem': 38,           'ef': 46},
}
_SEMS     = ['W0', 'W1', 'W2', 'W3']
_ALL_GEOS = ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro']
_CDR_GEO  = {'Mapapi': 'PB', 'NE Norte': 'PB', 'NE Sul': 'BA', 'NO Centro': 'PB'}

BIAS    = 1.09
DOI_MIN = 12
# Avaria NÃO aplicada — cabotagem não tem avaria

TI = {'MALZBIER': 25, 'GOOSE': 17, 'COLORADO': 33}
GS = {'MALZBIER': 20, 'GOOSE': 12, 'COLORADO': 28}

_CABO = {
    ('COLORADO','BA'): df_ct.iloc[3,4], ('COLORADO','PB'): df_ct.iloc[4,4],
    ('GOOSE',   'BA'): df_ct.iloc[5,4], ('GOOSE',   'PB'): df_ct.iloc[7,4],
    ('MALZBIER','BA'): df_ct.iloc[6,4], ('MALZBIER','PB'): df_ct.iloc[8,4],
}
MACO = {'MALZBIER': 285, 'GOOSE': 350, 'COLORADO': 300}

# EF W3 fevereiro — resultado do planejamento de fevereiro
EF_W3_FEV = {'MALZBIER': 31383, 'GOOSE': 38709, 'COLORADO': 13213}

# Demanda fev total c/BIAS e projeção março
dem_fev = {sku: sum(df_nd.iloc[TI[sku], _SC[s]['dem']] for s in _SEMS) / BIAS for sku in TI}
dem_mar = {sku: dem_fev[sku] * 1.10 / 4 for sku in TI}

rows = []
for sku in TI:
    rows.append({
        'SKU': sku,
        'Dem fev total c/BIAS (HL)':  round(dem_fev[sku], 1),
        '+10% → Dem mar total (HL)':  round(dem_fev[sku] * 1.10, 1),
        'Dem mar semanal (HL)':        round(dem_mar[sku], 1),
        'EI março = EF W3 fev (HL)':  EF_W3_FEV[sku],
    })
display(pd.DataFrame(rows))


,SKU,Dem fev total c/BIAS (HL),+10% → Dem mar total (HL),Dem mar semanal (HL),EI março = EF W3 fev (HL)
0,MALZBIER,"46,435.1","51,078.6","12,769.6",31383
1,GOOSE,"50,523.8","55,576.2","13,894.0",38709
2,COLORADO,"21,454.4","23,599.9","5,900.0",13213


## 3. Split BA/PB — Proporção por CDR

> Baseado na demanda de W0 de fevereiro (semana de referência para transferência).  
> NO Araguaia excluído — retirada em UB541/MG.


In [3]:
def get_prop_ba(sku, sem):
    g = GS[sku]
    ba = sum(df_nd.iloc[g+j, _SC[sem]['dem']] for j,geo in enumerate(_ALL_GEOS)
             if geo in _CDR_GEO and _CDR_GEO[geo]=='BA')
    pb = sum(df_nd.iloc[g+j, _SC[sem]['dem']] for j,geo in enumerate(_ALL_GEOS)
             if geo in _CDR_GEO and _CDR_GEO[geo]=='PB')
    return ba / (ba + pb)

prop_ba = {sku: get_prop_ba(sku, 'W0') for sku in TI}

display(pd.DataFrame([{
    'SKU': sku,
    'Prop. CDR Camaçari BA':    f'{prop_ba[sku]:.1%}',
    'Prop. CDR João Pessoa PB': f'{1-prop_ba[sku]:.1%}'
} for sku in TI]))
print('Referência: demanda W0 fevereiro por GEO, excluindo NO Araguaia')


,SKU,Prop. CDR Camaçari BA,Prop. CDR João Pessoa PB
0,MALZBIER,23.9%,76.1%
1,GOOSE,21.8%,78.2%
2,COLORADO,20.0%,80.0%


Referência: demanda W0 fevereiro por GEO, excluindo NO Araguaia


## 4. Balanço W0 Março — Volume de Cabotagem Necessário

> **EF s/ext** = EI março − Dem semanal março (PCP março desconhecido — sem produção no balanço)  
> **DOI s/ext** = EF s/ext / dem semanal × 6  
> **Transf. chegada = Embarque** — cabotagem não tem avaria


In [4]:
resultados = []

for sku in ['MALZBIER', 'GOOSE', 'COLORADO']:
    ei      = EF_W3_FEV[sku]
    dem     = dem_mar[sku]
    ef_sem  = ei - dem
    doi_sem = ef_sem / dem * 6
    transf  = max(0., DOI_MIN * dem / 6 - ef_sem)
    pb_ba   = prop_ba[sku]
    # Cabotagem: sem avaria — embarque = chegada
    emb_ba  = transf * pb_ba
    emb_pb  = transf * (1 - pb_ba)
    c_ba    = emb_ba * _CABO[(sku, 'BA')]
    c_pb    = emb_pb * _CABO[(sku, 'PB')]
    custo   = c_ba + c_pb
    ef_fin  = ef_sem + transf
    doi_fin = ef_fin / dem * 6
    resultados.append({
        'SKU': sku, 'EI (HL)': ei,
        'Dem semanal (HL)': round(dem, 1),
        'EF s/ext (HL)': round(ef_sem, 1),
        'DOI s/ext (d)': round(doi_sem, 1),
        '⚑': '✅' if doi_sem >= DOI_MIN else '🚨',
        'Transf./Emb. (HL)': round(transf, 1),
        'Emb. BA (HL)': round(emb_ba, 1),
        'Emb. PB (HL)': round(emb_pb, 1),
        'EF final (HL)': round(ef_fin, 1),
        'DOI final (d)': round(doi_fin, 1),
        '_custo': custo, '_c_ba': c_ba, '_c_pb': c_pb,
        '_emb_ba': emb_ba, '_emb_pb': emb_pb, '_transf': transf,
    })

cols = ['SKU','EI (HL)','Dem semanal (HL)','EF s/ext (HL)',
        'DOI s/ext (d)','⚑','Transf./Emb. (HL)',
        'Emb. BA (HL)','Emb. PB (HL)','EF final (HL)','DOI final (d)']
display(pd.DataFrame(resultados)[cols])

tot_transf = sum(r['_transf'] for r in resultados)
print(f'\nTotal embarque cabotagem (02/02): {tot_transf:,.1f} HL')
print(f'(embarque = chegada — sem avaria)')
print(f'\n📌 Todos os 3 SKUs abaixo de DOI 12d sem cabotagem.')
print(f'   Colorado, dispensado em fevereiro, já exige transferência em março.')


,SKU,EI (HL),Dem semanal (HL),EF s/ext (HL),DOI s/ext (d),⚑,Transf./Emb. (HL),Emb. BA (HL),Emb. PB (HL),EF final (HL),DOI final (d)
0,MALZBIER,31383,"12,769.6","18,613.4",8.7,🚨,"6,925.9","1,656.7","5,269.2","25,539.3",12.0
1,GOOSE,38709,"13,894.0","24,815.0",10.7,🚨,"2,973.1",647.3,"2,325.8","27,788.1",12.0
2,COLORADO,13213,"5,900.0","7,313.0",7.4,🚨,"4,486.9",896.6,"3,590.4","11,799.9",12.0



Total embarque cabotagem (02/02): 14,386.0 HL
(embarque = chegada — sem avaria)

📌 Todos os 3 SKUs abaixo de DOI 12d sem cabotagem.
   Colorado, dispensado em fevereiro, já exige transferência em março.


## 5. Custos de Cabotagem e Impacto no MACO

> MACO já inclui custo de produção — custo relevante é apenas o frete.


In [5]:
print('=== Custos Unitários de Cabotagem ===')
rows_cu = []
for sku in ['MALZBIER', 'GOOSE', 'COLORADO']:
    rows_cu.append({
        'SKU': sku,
        'Cabo BA (R$/HL)':       round(_CABO[(sku,'BA')], 2),
        'Cabo PB (R$/HL)':       round(_CABO[(sku,'PB')], 2),
        'MACO local (R$/HL)':    MACO[sku],
        'MACO líq. BA (R$/HL)':  round(MACO[sku] - _CABO[(sku,'BA')], 2),
        'MACO líq. PB (R$/HL)':  round(MACO[sku] - _CABO[(sku,'PB')], 2),
    })
display(pd.DataFrame(rows_cu))

print('\n=== Custo Total por SKU — W0 Março ===')
rows_ct = []
tot_c_ba = tot_c_pb = 0
for r in resultados:
    rows_ct.append({
        'SKU': r['SKU'],
        'Emb. BA (HL)':    round(r['_emb_ba'], 1),
        'Custo BA (R$)':   round(r['_c_ba'], 0),
        'Emb. PB (HL)':    round(r['_emb_pb'], 1),
        'Custo PB (R$)':   round(r['_c_pb'], 0),
        'Custo Total (R$)': round(r['_custo'], 0),
    })
    tot_c_ba += r['_c_ba']
    tot_c_pb += r['_c_pb']
display(pd.DataFrame(rows_ct))

tot_custo = tot_c_ba + tot_c_pb
tot_ba = sum(r['_emb_ba'] for r in resultados)
tot_pb = sum(r['_emb_pb'] for r in resultados)
print(f'\n  CDR Camaçari (BA):    {tot_ba:>7,.1f} HL  →  R${tot_c_ba:>10,.0f}')
print(f'  CDR João Pessoa (PB): {tot_pb:>7,.1f} HL  →  R${tot_c_pb:>10,.0f}')
print(f'  TOTAL CABOTAGEM W0:   {tot_ba+tot_pb:>7,.1f} HL  →  R${tot_custo:>10,.0f}')
print(f'\n⚠️  Custo refere-se apenas a W0 de março (1 semana).')
print(f'   Estimativa para o mês completo: ~R${tot_custo*4:,.0f} (4 semanas × mesmo volume).')


=== Custos Unitários de Cabotagem ===


,SKU,Cabo BA (R$/HL),Cabo PB (R$/HL),MACO local (R$/HL),MACO líq. BA (R$/HL),MACO líq. PB (R$/HL)
0,MALZBIER,84.6,95.3,285,200.4,189.7
1,GOOSE,82.4,88.3,350,267.6,261.7
2,COLORADO,76.6,82.1,300,223.4,217.9



=== Custo Total por SKU — W0 Março ===


,SKU,Emb. BA (HL),Custo BA (R$),Emb. PB (HL),Custo PB (R$),Custo Total (R$)
0,MALZBIER,"1,656.7","140,125.0","5,269.2","502,305.0","642,431.0"
1,GOOSE,647.3,"53,336.0","2,325.8","205,365.0","258,702.0"
2,COLORADO,896.6,"68,671.0","3,590.4","294,696.0","363,367.0"



  CDR Camaçari (BA):    3,200.6 HL  →  R$   262,132
  CDR João Pessoa (PB): 11,185.4 HL  →  R$ 1,002,367
  TOTAL CABOTAGEM W0:   14,386.0 HL  →  R$ 1,264,499

⚠️  Custo refere-se apenas a W0 de março (1 semana).
   Estimativa para o mês completo: ~R$5,057,996 (4 semanas × mesmo volume).


## 6. Análise de Inviabilidade — Por que Não Recomendamos


In [6]:
print('=' * 68)
print('POR QUE A EXPANSÃO VIA INCENTIVO COMERCIAL É INVIÁVEL A CURTO PRAZO')
print('=' * 68)
print('''
1. CAPACIDADE PRODUTIVA ESGOTADA
   Fevereiro já operou com folga zero em W2 e W3 (AQ541 + NS541 a 100%).
   Crescimento de +10%/mês supera qualquer capacidade de expansão local.
   Não há linha ociosa para absorver o incremento internamente.

2. CUSTO DE TRANSFERÊNCIA ESTRUTURAL E CRESCENTE
   Fevereiro: R$4,56M de rodo (emergência).
   Março W0: R$1,26M só de cabotagem — extrapolando: ~R$5,0M no mês.
   Com +10%/mês, esse custo cresce todo mês, corroendo o MACO sistematicamente.
   A cabotagem é mais barata que o rodo, mas não elimina o problema estrutural.

3. VISIBILIDADE RESTRITA — SÓ ENXERGAMOS O NENO
   A análise cobre exclusivamente GEOs NE e NO.
   Capacidade de cabotagem e estoque de SP podem estar comprometidos
   com outras regiões. Alocar tudo para NENO sem essa visibilidade
   pode gerar ruptura em outras GEOs.

4. RISCO DE NÃO REALIZAÇÃO DA DEMANDA
   O +10%/mês é sinalização comercial, não contrato.
   Se não realizar, o custo de transferência já foi incorrido
   e o estoque acumulado vira capital imobilizado.

5. RECOMENDAÇÃO DE MÉDIO PRAZO
   A única alavanca estrutural é expansão de capacidade produtiva no NENO
   (nova linha ou ampliação de AQ541/NS541).
   Enquanto isso, cabotagem é solução transitória — não permanente.
''')


POR QUE A EXPANSÃO VIA INCENTIVO COMERCIAL É INVIÁVEL A CURTO PRAZO

1. CAPACIDADE PRODUTIVA ESGOTADA
   Fevereiro já operou com folga zero em W2 e W3 (AQ541 + NS541 a 100%).
   Crescimento de +10%/mês supera qualquer capacidade de expansão local.
   Não há linha ociosa para absorver o incremento internamente.

2. CUSTO DE TRANSFERÊNCIA ESTRUTURAL E CRESCENTE
   Fevereiro: R$4,56M de rodo (emergência).
   Março W0: R$1,26M só de cabotagem — extrapolando: ~R$5,0M no mês.
   Com +10%/mês, esse custo cresce todo mês, corroendo o MACO sistematicamente.
   A cabotagem é mais barata que o rodo, mas não elimina o problema estrutural.

3. VISIBILIDADE RESTRITA — SÓ ENXERGAMOS O NENO
   A análise cobre exclusivamente GEOs NE e NO.
   Capacidade de cabotagem e estoque de SP podem estar comprometidos
   com outras regiões. Alocar tudo para NENO sem essa visibilidade
   pode gerar ruptura em outras GEOs.

4. RISCO DE NÃO REALIZAÇÃO DA DEMANDA
   O +10%/mês é sinalização comercial, não contrato.
  

## 7. Resumo Executivo — Março W0

| Dimensão | Resultado |
|---|---|
| Volume cabotagem a embarcar em 02/02 | **14.386 HL** |
| Custo total cabotagem W0 | **R$ 1,26M** |
| SKUs que precisam de transferência | **Todos (Malzbier, Goose, Colorado)** |
| SKUs com DOI ok sem transferência | **Nenhum** |
| CDR Camaçari BA | 3.201 HL → R$ 262k |
| CDR João Pessoa PB | 11.185 HL → R$ 1.002k |
| MACO positivo com cabo? | **Sim** — mas margem reduzida e tendência de piora |
| Recomendação | **Não expandir via incentivo** sem expansão produtiva |

> ⚠️ Esta simulação cobre apenas W0 de março.  
> Nas semanas seguintes, sem PCP definido, o problema se repete ou piora.
